# Vision Assignment P02 — Pemrosesan Citra

Pipeline: grayscale → Gaussian blur → Canny dengan beberapa threshold → HSV/Lab → segmentasi warna.

In [ ]:
%pip install -q opencv-python-headless matplotlib numpy

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Gambar sintetis agar notebook reproducible tanpa upload file
image = np.zeros((256, 384, 3), dtype=np.uint8)
image[:] = (35, 35, 35)
cv2.rectangle(image, (45, 55), (180, 205), (40, 180, 230), -1)
cv2.circle(image, (285, 130), 70, (40, 60, 220), -1)
cv2.line(image, (20, 235), (360, 235), (220, 220, 220), 4)
rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)
hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
thresholds = [(30, 100), (50, 150), (100, 200)]
edges = [cv2.Canny(blur, low, high) for low, high in thresholds]

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(16, 7))
ax[0,0].imshow(rgb); ax[0,0].set_title('Original')
ax[0,1].imshow(gray, cmap='gray'); ax[0,1].set_title('Grayscale')
ax[0,2].imshow(blur, cmap='gray'); ax[0,2].set_title('Gaussian blur')
for a, e, t in zip(ax[0,3:], edges[:1], thresholds[:1]):
    a.imshow(e, cmap='gray'); a.set_title(f'Canny {t}')
for a, e, t in zip(ax[1,:3], edges, thresholds):
    a.imshow(e, cmap='gray'); a.set_title(f'Canny {t}')
ax[1,3].imshow(hsv[:,:,0], cmap='hsv'); ax[1,3].set_title('HSV Hue')
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Segmentasi objek merah: HSV OpenCV memakai H 0–179
mask = cv2.inRange(hsv, np.array([0, 80, 50]), np.array([10, 255, 255]))
segmented = cv2.bitwise_and(rgb, rgb, mask=mask)
plt.figure(figsize=(8,3))
plt.subplot(1,2,1); plt.imshow(mask, cmap='gray'); plt.title('Mask merah'); plt.axis('off')
plt.subplot(1,2,2); plt.imshow(segmented); plt.title('Hasil segmentasi'); plt.axis('off')
plt.show()
print('Edge pixels:', [int(np.count_nonzero(e)) for e in edges])
print('HSV shape:', hsv.shape, 'Lab shape:', lab.shape)

## Analisis, Interpretasi, dan Kesimpulan

Threshold rendah mendeteksi lebih banyak edge sekaligus lebih sensitif terhadap noise. Threshold tinggi lebih selektif tetapi dapat menghilangkan detail lemah. Gaussian blur membantu mengurangi edge palsu, tetapi blur berlebihan dapat menghilangkan detail.

HSV memudahkan pemilihan warna karena hue dipisahkan dari brightness; Lab menyediakan representasi yang lebih mendekati persepsi warna. Pada gambar nyata, threshold perlu disesuaikan dengan pencahayaan dan kamera, lalu divalidasi melalui visualisasi mask atau ground truth.

In [ ]:
# Kesimpulan otomatis berdasarkan hasil aktual
edge_counts = {str(t): int(np.count_nonzero(e)) for t, e in zip(thresholds, edges)}
print('ANALISIS OTOMATIS')
for threshold, count in edge_counts.items(): print(f'- Canny {threshold}: {count} edge pixels')
print(f'- Rasio mask merah: {np.count_nonzero(mask)/mask.size:.2%}')
print('KESIMPULAN: gunakan threshold yang menghasilkan kontur paling informatif dengan noise yang dapat diterima.')